In [1]:
#!/usr/bin/env python
# coding: utf-8
# Train the diffusion-style text model on TinyStories (or TinyShakespeare via a switch)

import os
import json
import math
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm import tqdm

# ========= Config =========
config = {
    # ---- dataset options ----
    'dataset_name': 'tinystories',     # 'tinystories' | 'tinyshakespeare'
    'hf_dataset': 'roneneldan/TinyStories',
    'hf_split': 'train',               # TinyStories: 'train' or 'validation'
    'max_chars': 30_000_000,           # cap to ~30M chars to avoid huge RAM
    'min_story_len': 128,              # skip very short stories
    'local_tinystories_path': None,    # optional: path to local JSONL with {"text": "..."} lines

    # ---- model/training ----
    'block_size': 512,
    'batch_size': 128,
    'embedding_dim': 512,
    'num_heads': 8,
    'num_layers': 8,
    'num_timesteps': 200,
    'learning_rate': 3e-4,
    'weight_decay': 0.01,
    'max_iters': 20000,
    'eval_interval': 500,
    'save_interval': 2000,
    'gradient_accumulation_steps': 2,
    'mixed_precision': True,
    'grad_clip': 1.0,
    'seed': 1337,
    'output_dir': './tinystories_checkpoints'
}

# ========= Setup =========
os.makedirs(config['output_dir'], exist_ok=True)

torch.manual_seed(config['seed'])
random.seed(config['seed'])
np.random.seed(config['seed'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
amp_enabled = (device == 'cuda') and bool(config['mixed_precision'])
print(f"Using device: {device} | AMP: {amp_enabled}")

# ========= Data loading =========
def load_tinyshakespeare_text():
    import requests
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    local = "input.txt"
    if not os.path.exists(local):
        print("Downloading tinyshakespeare…")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(local, "wb") as f:
            f.write(r.content)
    with open(local, 'r', encoding='utf-8') as f:
        return f.read()

def iter_local_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                txt = obj.get("text", "")
                if txt:
                    yield txt
            except Exception:
                continue

def load_tinystories_text(cfg):
    """
    Returns a single large string composed of TinyStories texts,
    capped by cfg['max_chars'] for RAM friendliness.
    """
    texts = []
    total = 0

    if cfg['local_tinystories_path']:
        print(f"Loading TinyStories from local JSONL: {cfg['local_tinystories_path']}")
        src = iter_local_jsonl(cfg['local_tinystories_path'])
        for s in src:
            s = s.strip()
            if len(s) < cfg['min_story_len']:
                continue
            s = s + "\n\n"
            texts.append(s)
            total += len(s)
            if total >= cfg['max_chars']:
                break
    else:
        try:
            from datasets import load_dataset
        except Exception as e:
            raise RuntimeError(
                "The 'datasets' package is required. Install with: pip install datasets"
            ) from e

        print(f"Loading TinyStories from HF: {cfg['hf_dataset']} [{cfg['hf_split']}]")
        ds = load_dataset(cfg['hf_dataset'], split=cfg['hf_split'])
        idxs = list(range(len(ds)))
        random.shuffle(idxs)
        for i in idxs:
            s = ds[i]['text']
            if not s:
                continue
            s = s.strip()
            if len(s) < cfg['min_story_len']:
                continue
            s = s + "\n\n"
            texts.append(s)
            total += len(s)
            if total >= cfg['max_chars']:
                break

    text = "".join(texts)
    print(f"TinyStories collected chars: {len(text):,} (cap={cfg['max_chars']:,})")
    return text

if config['dataset_name'].lower() == 'tinystories':
    text = load_tinystories_text(config)
else:
    text = load_tinyshakespeare_text()

print(f"Dataset total length: {len(text):,} characters")

# ========= Char tokenizer =========
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size}")

with open(os.path.join(config['output_dir'], 'tokenizer.json'), 'w') as f:
    json.dump({'chars': chars, 'vocab_size': vocab_size}, f, ensure_ascii=False)

# ========= Numeric data & split =========
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train chars: {len(train_data):,} | Val chars: {len(val_data):,}")

# ========= Dataset & batcher =========
class TokenDataset(Dataset):
    def __init__(self, data_tensor, block_size):
        self.data = data_tensor
        self.block_size = block_size
        if len(self.data) <= self.block_size:
            raise ValueError("Data is shorter than block_size.")
    def __len__(self):
        return len(self.data) // self.block_size
    def __getitem__(self, idx):
        start = idx * self.block_size
        chunk = self.data[start:start+self.block_size+1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

def get_batch(split, batch_size):
    src = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(src) - config['block_size'] - 1, (batch_size,))
    x = torch.stack([src[i:i+config['block_size']] for i in ix])
    y = torch.stack([src[i+1:i+config['block_size']+1] for i in ix])
    return x, y

# ========= Model =========
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        emb_factor = math.log(10000) / max(half_dim - 1, 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb_factor)
        emb = time[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        if emb.shape[-1] < self.dim:
            emb = torch.nn.functional.pad(emb, (0, self.dim - emb.shape[-1]))
        return emb

class ShakespeareDiffusion(nn.Module):
    """
    Categorical diffusion-like token masker with a Transformer denoiser.
    """
    def __init__(self, vocab_size, config):
        super().__init__()
        self.vocab_size = vocab_size
        self.config = config

        self.token_embedding = nn.Embedding(vocab_size, config['embedding_dim'])
        self.position_embedding = nn.Embedding(config['block_size'], config['embedding_dim'])

        self.time_embedding = nn.Sequential(
            SinusoidalPositionEmbeddings(config['embedding_dim']),
            nn.Linear(config['embedding_dim'], config['embedding_dim'] * 4),
            nn.GELU(),
            nn.Linear(config['embedding_dim'] * 4, config['embedding_dim'])
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config['embedding_dim'],
            nhead=config['num_heads'],
            dim_feedforward=config['embedding_dim'] * 4,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, config['num_layers'])

        self.ln_out = nn.LayerNorm(config['embedding_dim'])
        self.output_proj = nn.Linear(config['embedding_dim'], vocab_size)

        self.register_buffer('beta', self._cosine_beta_schedule())
        self.register_buffer('alpha', 1. - self.beta)
        self.register_buffer('alpha_bar', torch.cumprod(self.alpha, dim=0))

    def _cosine_beta_schedule(self, s=0.008):
        steps = torch.linspace(0, self.config['num_timesteps'], self.config['num_timesteps'] + 1)
        alpha_bar = torch.cos(((steps / self.config['num_timesteps']) + s) / (1 + s) * math.pi / 2) ** 2
        beta = 1 - alpha_bar[1:] / alpha_bar[:-1]
        return torch.clip(beta, 0.0001, 0.9999)

    def forward(self, x, t, targets=None):
        B, T = x.shape
        if T > self.config['block_size']:
            raise ValueError("Sequence length exceeds block_size configured for position embeddings.")

        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(torch.arange(T, device=x.device))
        time_emb = self.time_embedding(t).unsqueeze(1).expand(-1, T, -1)

        x_emb = token_emb + pos_emb + time_emb
        x_transformed = self.transformer(x_emb)
        x_transformed = self.ln_out(x_transformed)
        logits = self.output_proj(x_transformed)

        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))
            return logits, loss
        return logits

    def q_sample(self, x0, t, noise=None):
        # Categorical forward process: replace tokens with random tokens with prob (1 - alpha_bar_t)
        if noise is None:
            noise = torch.randint(0, self.vocab_size, x0.shape, device=x0.device)
        alpha_bar_t = self.alpha_bar[t].view(-1, 1)
        mask_prob = 1 - alpha_bar_t
        mask = torch.rand_like(x0.float()) < mask_prob
        xt = torch.where(mask, noise, x0)
        return xt

    @torch.no_grad()
    def p_sample(self, xt, t, temperature=1.0):
        logits = self(xt, t)
        logits = logits / max(temperature, 1e-5)
        probs = F.softmax(logits, dim=-1)
        x_next = torch.multinomial(probs.view(-1, self.vocab_size), 1)
        x_next = x_next.view(xt.shape)
        return x_next

    def generate(self, prompt=None, length=200, num_steps=50, temperature=0.8):
        self.eval()
        device = next(self.parameters()).device
        if prompt is None:
            B = 1
            xt = torch.randint(0, self.vocab_size, (B, length), device=device)
        else:
            prompt = prompt.to(device)
            B, prompt_len = prompt.shape
            if length < prompt_len:
                raise ValueError("length must be >= prompt length")
            xt = torch.randint(0, self.vocab_size, (B, length), device=device)
            xt[:, :prompt_len] = prompt

        step = max(1, self.config['num_timesteps'] // num_steps)
        for t in reversed(range(0, self.config['num_timesteps'], step)):
            t_batch = torch.full((B,), t, device=device, dtype=torch.long)
            xt = self.p_sample(xt, t_batch, temperature)
        return xt

# ========= Utils =========
def save_checkpoint(model, optimizer, global_step, cfg, path):
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'global_step': global_step,
        'config': cfg
    }, path)
    print(f"Checkpoint saved to {path}")

def preview_1line(s, n=120):
    # Avoid backslashes inside f-string expressions by precomputing the preview
    return s[:n].replace("\n", " ")

# ========= Training =========
def train_model():
    model = ShakespeareDiffusion(vocab_size, config).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params / 1e6:.2f}M")

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=config['learning_rate'],
                                  weight_decay=config['weight_decay'])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['max_iters'], eta_min=1e-5
    )

    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    global_step = 0
    training_log = []

    pbar = tqdm(total=config['max_iters'], desc="Training TinyStories Diffusion")

    for it in range(config['max_iters']):
        model.train()
        xb, yb = get_batch('train', config['batch_size'])
        xb, yb = xb.to(device), yb.to(device)
        t = torch.randint(0, config['num_timesteps'], (xb.shape[0],), device=device)

        # forward diffusion
        noise = torch.randint(0, vocab_size, xb.shape, device=device)
        xt = model.q_sample(xb, t, noise)

        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits, loss = model(xt, t, yb)
            loss = loss / config['gradient_accumulation_steps']

        scaler.scale(loss).backward()

        if (it + 1) % config['gradient_accumulation_steps'] == 0:
            if config['grad_clip'] is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            pbar.update(1)

        # eval & samples
        if it % config['eval_interval'] == 0:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for _ in range(10):
                    xb_val, yb_val = get_batch('val', config['batch_size'])
                    xb_val, yb_val = xb_val.to(device), yb_val.to(device)
                    t_val = torch.randint(0, config['num_timesteps'], (xb_val.shape[0],), device=device)
                    noise_val = torch.randint(0, vocab_size, xb_val.shape, device=device)
                    xt_val = model.q_sample(xb_val, t_val, noise_val)
                    _, loss_val = model(xt_val, t_val, yb_val)
                    val_losses.append(loss_val.item())
            val_loss = float(np.mean(val_losses))

            with torch.no_grad():
                # from scratch
                sample = model.generate(length=min(300, config['block_size']), num_steps=50, temperature=0.7)
                gen_text = decode(sample[0].cpu().tolist())
                # from prompt
                prompt = "Once upon a time "
                prompt_tokens = torch.tensor([[stoi.get(c, 0) for c in prompt]], dtype=torch.long, device=device)
                sample2 = model.generate(prompt=prompt_tokens, length=min(300, config['block_size']), num_steps=50, temperature=0.7)
                gen_prompt_text = decode(sample2[0].cpu().tolist())

            # precompute previews to avoid backslashes in f-string expressions
            gen_preview = preview_1line(gen_text, 120)
            prompt_preview = preview_1line(gen_prompt_text, 120)

            log_entry = {
                'step': it,
                'train_loss': float((loss.item() * config['gradient_accumulation_steps'])),
                'val_loss': val_loss,
                'lr': float(scheduler.get_last_lr()[0]),
                'generated_sample': gen_text[:400],
                'generated_prompt': gen_prompt_text[:400]
            }
            training_log.append(log_entry)
            print(f"\nStep {it} | Train {log_entry['train_loss']:.4f} | Val {val_loss:.4f} | LR {log_entry['lr']:.6f}")
            print(f"Gen: {gen_preview} …")
            print(f"Prompt 'Once upon a time': {prompt_preview} …")
            model.train()

        if it % config['save_interval'] == 0:
            save_checkpoint(model, optimizer, it, config,
                            os.path.join(config['output_dir'], f'checkpoint_{it}.pt'))

    pbar.close()

    # final save
    save_checkpoint(model, optimizer, config['max_iters'], config,
                    os.path.join(config['output_dir'], 'final_model.pt'))

    with open(os.path.join(config['output_dir'], 'training_log.json'), 'w') as f:
        json.dump(training_log, f, indent=2)

    return model

def generate_text(model, prompt=None, length=200, temperature=0.8):
    model.eval()
    with torch.no_grad():
        if prompt is None:
            sample = model.generate(length=min(length, config['block_size']), num_steps=50, temperature=temperature)
        else:
            prompt_tokens = torch.tensor([[stoi.get(c, 0) for c in prompt]], dtype=torch.long, device=device)
            sample = model.generate(prompt=prompt_tokens, length=min(length, config['block_size']), num_steps=50, temperature=temperature)
        return decode(sample[0].cpu().tolist())

# ========= Main =========
if __name__ == "__main__":
    print("Starting TinyStories training…")
    print(f"Config: {json.dumps(config, indent=2)}")
    model = train_model()

    print("\nTraining complete! Sample generations:")
    for p in [None, "Once upon a time ", "The little robot ", "A tiny dragon "]:
        label = "from scratch" if p is None else f"prompt '{p}'"
        print(f"\nGenerated {label}:")
        text_out = generate_text(model, prompt=p, length=300, temperature=0.7)
        print(text_out)


Using device: cuda | AMP: True
Loading TinyStories from HF: roneneldan/TinyStories [train]
TinyStories collected chars: 30,000,002 (cap=30,000,000)
Dataset total length: 30,000,002 characters
Vocabulary size: 98
Train chars: 27,000,001 | Val chars: 3,000,001
Starting TinyStories training…
Config: {
  "dataset_name": "tinystories",
  "hf_dataset": "roneneldan/TinyStories",
  "hf_split": "train",
  "max_chars": 30000000,
  "min_story_len": 128,
  "local_tinystories_path": null,
  "block_size": 512,
  "batch_size": 128,
  "embedding_dim": 512,
  "num_heads": 8,
  "num_layers": 8,
  "num_timesteps": 200,
  "learning_rate": 0.0003,
  "weight_decay": 0.01,
  "max_iters": 20000,
  "eval_interval": 500,
  "save_interval": 2000,
  "gradient_accumulation_steps": 2,
  "mixed_precision": true,
  "grad_clip": 1.0,
  "seed": 1337,
  "output_dir": "./tinystories_checkpoints"
}


/opt/conda/lib/python3.10/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Model parameters: 27.68M


Training TinyStories Diffusion:   0%|          | 0/20000 [00:00<?, ?it/s]


Step 0 | Train 4.8331 | Val 4.8366 | LR 0.000300
Gen: 80/XWHeh+hFAM“W3œs‹Ãi3j:9)Hj¯©E8ÃE !C>œ0d0dJ¦mœlMH/-RsCC/˜6DLqL¯Y;<_C~yIpœsy5s™œ‹5±pQj¦p~_™m.ac7nU™DFu4V>El©)™g&s3˜Qd¯‹H …
Prompt 'Once upon a time': (H,<V5$hi-HdZ4oBQp/)fld“z™˜MkdjgDXd_lym,9yvpVV™y<gœq_j+mCSLI/rlW 5Id"N Xeo œWsBâQgY±NJJ9_9,‹Q"l“S©GHVSNJUœ8,_) `jINH5oCp …
Checkpoint saved to ./tinystories_checkpoints/checkpoint_0.pt


...


Step 19000 | Train 1.2403 | Val 1.2538 | LR 0.000166
Gen: ee said too. Tom and on the big boy heard the castle and then the the boy said they got he had the per. The bug back a b …
Prompt 'Once upon a time': e the beard togethe the red fun with the water and the care in the boat the to the teathings to the sea to the said then …


Training TinyStories Diffusion:  49%|████▉     | 9751/20000 [55:24<3:10:19,  1.11s/it]


Step 19500 | Train 1.3753 | Val 1.2303 | LR 0.000161
Gen:  game day, that he ran of the but the bite to the farmer the gave there was sad and try to her and them and that to the  …
Prompt 'Once upon a time':  to that the day, they felt in the toy water and the back of steat liked the store the child not would pets of the mooti …


Training TinyStories Diffusion:  50%|█████     | 10000/20000 [56:44<56:44,  2.94it/s] 


Checkpoint saved to ./tinystories_checkpoints/final_model.pt

Training complete! Sample generations:

Generated from scratch:
e the boy and the yard and no. They made a letter. Then the toat the beard hers and walked and the fore who had a put him and to her toy boy and the boy and scared there was a little bird the big boy. They saround the bead and in her had tried to worry and had a boy. The boy wer toy words and saw a 

Generated prompt 'Once upon a time ':
 want they see them to the big and that hey ate the day, they ran to the parches.

They are to warm to the answer in the door to ear and see the tell the park and pink. They were he mad. The the took the bag and them at them and in the reed the door and they are to the toy toys the to them, aterted 

Generated prompt 'The little robot ':
ey had toys. Then the toy the did not want it to the wanted they had a big park to the big house. They and be the big that to the tan ball and Lily. They are on the land and then she did to the 

In [2]:
def train_model(resume_from=None, extra_iters=10000):
    """
    If resume_from is provided (path to .pt), continue training for `extra_iters` micro-iterations.
    Otherwise, start fresh and run for config['max_iters'] micro-iterations.

    Notes:
    - We saved 'global_step' (optimizer steps) in the checkpoint.
    - One optimizer step happens every `gradient_accumulation_steps` micro-iterations.
    """
    model = ShakespeareDiffusion(vocab_size, config).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    # Defaults for a fresh run
    start_opt_steps = 0
    start_micro = 0
    target_micro = config['max_iters']  # by default

    # ---- Load checkpoint if resuming ----
    if resume_from is not None and os.path.isfile(resume_from):
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt['model'])
        try:
            optimizer.load_state_dict(ckpt['optimizer'])
        except Exception as e:
            print(f"Warning: optimizer state load failed ({e}); continuing with fresh optimizer.")

        start_opt_steps = int(ckpt.get('global_step', 0))
        start_micro = start_opt_steps * config['gradient_accumulation_steps']
        target_micro = start_micro + int(extra_iters)

        # Make sure LR in optimizer matches current config (some checkpoints may store older LR)
        for g in optimizer.param_groups:
            g['lr'] = config['learning_rate']

        print(f"Resuming from: {resume_from}")
        print(f"  start_opt_steps: {start_opt_steps}")
        print(f"  start_micro:     {start_micro}")
        print(f"  train more:      {extra_iters} micro-iters (to {target_micro})")
    else:
        print("Starting fresh training run.")
        # fresh: run whatever is in config['max_iters']

    # Recreate scheduler at the right point in the cosine cycle.
    # We call scheduler.step() at every optimizer step, so last_epoch should be the number of steps done.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=target_micro // config['gradient_accumulation_steps'] if resume_from else config['max_iters'],
        eta_min=1e-5,
        last_epoch=start_opt_steps  # resumes LR where you left off
    )

    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    # For logging
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params / 1e6:.2f}M")

    global_step = start_opt_steps
    training_log = []

    pbar_total = target_micro - start_micro
    pbar = tqdm(total=pbar_total, desc="Training TinyStories Diffusion (resumable)")

    # Main loop: continue from start_micro up to target_micro
    for it in range(start_micro, target_micro):
        model.train()
        xb, yb = get_batch('train', config['batch_size'])
        xb, yb = xb.to(device), yb.to(device)
        t = torch.randint(0, config['num_timesteps'], (xb.shape[0],), device=device)

        # forward diffusion
        noise = torch.randint(0, vocab_size, xb.shape, device=device)
        xt = model.q_sample(xb, t, noise)

        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits, loss = model(xt, t, yb)
            loss = loss / config['gradient_accumulation_steps']

        scaler.scale(loss).backward()

        # Optimizer step on GA boundary
        if (it + 1) % config['gradient_accumulation_steps'] == 0:
            if config.get('grad_clip', None) is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

        # Progress bar always counts micro-iterations
        pbar.update(1)

        # Eval & samples (uses absolute micro-iter index 'it' for frequency)
        if it % config['eval_interval'] == 0:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for _ in range(10):
                    xb_val, yb_val = get_batch('val', config['batch_size'])
                    xb_val, yb_val = xb_val.to(device), yb_val.to(device)
                    t_val = torch.randint(0, config['num_timesteps'], (xb_val.shape[0],), device=device)
                    noise_val = torch.randint(0, vocab_size, xb_val.shape, device=device)
                    xt_val = model.q_sample(xb_val, t_val, noise_val)
                    _, loss_val = model(xt_val, t_val, yb_val)
                    val_losses.append(loss_val.item())
            val_loss = float(np.mean(val_losses))

            with torch.no_grad():
                # from scratch
                sample = model.generate(length=min(300, config['block_size']), num_steps=50, temperature=0.7)
                gen_text = decode(sample[0].cpu().tolist())
                # from prompt
                prompt = "Once upon a time "
                prompt_tokens = torch.tensor([[stoi.get(c, 0) for c in prompt]], dtype=torch.long, device=device)
                sample2 = model.generate(prompt=prompt_tokens, length=min(300, config['block_size']), num_steps=50, temperature=0.7)
                gen_prompt_text = decode(sample2[0].cpu().tolist())

            gen_preview = preview_1line(gen_text, 120)
            prompt_preview = preview_1line(gen_prompt_text, 120)

            log_entry = {
                'step': it,
                'train_loss': float(loss.item() * config['gradient_accumulation_steps']),
                'val_loss': val_loss,
                'lr': float(scheduler.get_last_lr()[0]),
                'generated_sample': gen_text[:400],
                'generated_prompt': gen_prompt_text[:400]
            }
            training_log.append(log_entry)
            print(f"\nStep {it} | Train {log_entry['train_loss']:.4f} | Val {val_loss:.4f} | LR {log_entry['lr']:.6f}")
            print(f"Gen: {gen_preview} …")
            print(f"Prompt 'Once upon a time': {prompt_preview} …")
            model.train()

        # Periodic save (keeps new global_step)
        if it % config['save_interval'] == 0:
            save_checkpoint(
                model, optimizer, global_step, config,
                os.path.join(config['output_dir'], f'checkpoint_{it}.pt')
            )

    pbar.close()

    # Final save after resume
    save_checkpoint(
        model, optimizer, global_step, config,
        os.path.join(config['output_dir'], 'final_model.pt')
    )

    with open(os.path.join(config['output_dir'], 'training_log.json'), 'w') as f:
        json.dump(training_log, f, indent=2)

    return model


In [3]:
model = ShakespeareDiffusion(vocab_size, config).to(device)
resume_from="./tinystories_checkpoints/final_model.pt"
# ---- Load checkpoint if resuming ----
if resume_from is not None and os.path.isfile(resume_from):
    ckpt = torch.load(resume_from, map_location=device)
    model.load_state_dict(ckpt['model'])
    

/opt/conda/lib/python3.10/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [10]:
for p in [None, "Once upon a time ", "The little robot ", "A tiny dragon "]:
    label = "from scratch" if p is None else f"prompt '{p}'"
    print(f"\nGenerated {label}:")
    text_out = generate_text(model, prompt=p, length=300, temperature=0.7)
    print(text_out)



Generated from scratch:
 get to sand and dad and the sand the should the do and heared the pee in the sand it go in their and decided the closer the to the sand. The sand man and slide the treat. The sand the sand day the red to see in the started the the share some into them, the sand into a seen and seen day and red tond

Generated prompt 'Once upon a time ':
the brother and said to go that he was they liked to see thered if them and had her fun. The hool on the they wanted to the hool. The little the mittings. The end the are them got to the strong asked than the said them about the hold her liked them. They are the sun what be the hool and red them ath

Generated prompt 'The little robot ':
o see the reached they loved to wail. The hands if them them. They heard see their dream them, the hones and place the room and see them then them as they had. The do to them to them to scared and them when the do them them to then they are no and then he was their old them scared them are on t

In [ ]:
model = train_model(resume_from="./tinystories_checkpoints/final_model.pt", extra_iters=10000)


Resuming from: ./tinystories_checkpoints/final_model.pt
  start_opt_steps: 25000
  start_micro:     50000
  train more:      10000 micro-iters (to 60000)
Model parameters: 27.68M


Training TinyStories Diffusion (resumable):   0%|          | 0/10000 [00:00<?, ?it/s]


Step 50000 | Train 1.3721 | Val 1.1989 | LR 0.000029
Gen: cat ran to the cat cat. He back to the park. The cat the cat to the cat. They mad and see the birds and them to see the  …
Prompt 'Once upon a time': ke her toy had she saw the sing and had at her and saw she hope. She in the way that she waved there she got that made w …


Training TinyStories Diffusion (resumable):   0%|          | 2/10000 [00:03<4:16:27,  1.54s/it]

Checkpoint saved to ./tinystories_checkpoints/checkpoint_50000.pt


...

Training TinyStories Diffusion (resumable):  80%|████████  | 8000/10000 [22:50<05:30,  6.05it/s]


Step 58000 | Train 1.3543 | Val 1.2576 | LR 0.000011
Gen: beacher mom in the bear. The stop and then she was so hap in the was stay. They how the way to the other and the sky. Sh …
Prompt 'Once upon a time':  bones. The other. The boat up on the saw them and the he reached the liked the boat the boat they got in the big boat t …


Training TinyStories Diffusion (resumable):  80%|████████  | 8002/10000 [22:54<20:19,  1.64it/s]

Checkpoint saved to ./tinystories_checkpoints/checkpoint_58000.pt


Training TinyStories Diffusion (resumable):  83%|████████▎ | 8276/10000 [23:39<04:48,  5.98it/s]